# 06 Token 精确计数 + 上下文裁剪

之前用 `total_chars // 2` 粗估 token，对中文严重低估（1 字符 ≈ 1.5 token）对英文高估（1 字符 ≈ 0.25 token）。
06 引入 `tiktoken` 做精确计数，并支持超限自动裁剪。

**学习点**：token 不是字符，中文和英文的 tokenization 完全不同。

In [ ]:
import sys; sys.path.insert(0, '..')
from src.agent_framework.memory import ConversationMemory

print('一切就绪')

## chars//2 vs tiktoken 误差

In [ ]:
test_cases = [
    ("纯英文", "Hello, how are you today? I would like to learn more about AI." * 5),
    ("纯中文", "人工智能是计算机科学的一个分支，致力于创造能够模拟人类智能的系统。" * 5),
    ("混合", "今天AI技术发展很快，LLM模型越来越强大，DeepSeek是很好的选择。" * 5),
    ("代码", "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)" * 5),
]

for label, text in test_cases:
    mem = ConversationMemory()
    mem.add_user(text)
    real = mem.token_count()
    rough = len(text) // 2
    err = (real - rough) / real * 100
    print(f'{label}: tiktoken={real}, chars//2={rough}, 误差={err:.0f}%')"

## 自动裁剪

In [ ]:
mem = ConversationMemory(max_tokens=150)
mem.add_system('你是一个有用的AI助手，请用中文回复。')

for i in range(30):
    mem.add_user(f'第{i}条消息：今天天气真好啊，适合出去走走，不知道公园里人会不会很多')
    mem.add_assistant(type('msg',(),{'content': f'是的第{i}条消息收到了，建议去附近的公园看看','tool_calls':None})())

s = mem.stats()
print(f'30轮对话后: {s["n_messages"]} 条消息, {s["tokens"]} tokens (上限150)')
print(f'system prompt 保留: {mem._messages[0]["role"] == "system"}')

## 对比新旧 API

In [ ]:
import sys, os, shutil
sys.path.insert(0, '..')
for d in ['agent_memory']:
    if os.path.exists(d): shutil.rmtree(d)

from src.agent_framework import Agent, EmbeddingStore
from src.capabilities import LongTermMemory, PlanManager
from src.capabilities.demo_tools import create_demo_tools

es = EmbeddingStore()
ltm = LongTermMemory(embedding_store=es)
plan_mgr = PlanManager()
tools = create_demo_tools(plan_mgr=plan_mgr, ltm=ltm)

agent = Agent(
    tools=tools, long_term_memory=ltm, plan_mgr=plan_mgr,
    tool_top_k=5, embedding_store=es, max_tokens=2000,
)

print(f'Agent 启动: max_tokens={agent.memory.max_tokens}')
print(f'初始 token: {agent.memory.token_count()}')

reply = agent.chat('你好，我叫小明')
print(f'对话后 token: {agent.memory.token_count()}')

agent.memory.clear()
shutil.rmtree('agent_memory')
print('清理完成')